# KMP 字符串匹配算法

KMP（Knuth-Morris-Pratt）用于在主串 `text` 中查找模式串 `pattern` 出现的位置。

它解决的问题是：当匹配到一半发生失配时，朴素算法会反复比较已经确认过的字符；KMP 利用模式串自身的结构，避免主串指针回退。

## 1. KMP 的动机

先看朴素字符串匹配。假设要在 `text` 中查找 `pattern`，朴素做法会枚举每一个可能的起点 `start`，然后从头比较模式串：

```text
text:    a a a a a b
pattern: a a a b
         ^ ^ ^ x
```

在起点 `0`，前三个字符都匹配，最后一个字符失配。朴素算法会把起点右移一位，然后重新从 `pattern[0]` 开始比较：

```text
text:      a a a a a b
pattern:   a a a b
           ^ ^ ^ x
```

问题在于：前面已经比较过很多 `a`，但朴素算法没有利用这些信息。

如果 `text = "aaaaaaaaab"`，`pattern = "aaaab"`，很多起点都会先成功匹配多个 `a`，再在 `b` 处失败。因此最坏时间复杂度是：

$$
O(nm)
$$

其中 `n = len(text)`，`m = len(pattern)`。

KMP 的核心目标是：

> 主串指针 `i` 永不回退；失配时只移动模式串指针 `j`。

### 朴素匹配实现

下面的函数返回 `pattern` 在 `text` 中出现的所有起始下标。

In [ ]:
def naive_search(text: str, pattern: str) -> list[int]:
    """返回 pattern 在 text 中出现的所有起始下标。"""
    n, m = len(text), len(pattern)

    if m == 0:
        return list(range(n + 1))

    matches = []

    for start in range(n - m + 1):
        j = 0
        while j < m and text[start + j] == pattern[j]:
            j += 1

        if j == m:
            matches.append(start)

    return matches

In [ ]:
naive_search("ABABDABACDABABCABAB", "ABABCABAB")

## 2. LPS 数组

KMP 的关键预处理结构是 `LPS` 数组。

`LPS` 是 **Longest Proper Prefix which is also Suffix** 的缩写。

对模式串 `pattern` 的每一个位置 `i`，`lps[i]` 表示：

> 在子串 `pattern[0:i+1]` 中，最长的、既是前缀又是后缀的真前缀长度。

这里的“真前缀”表示不能等于字符串本身。

例如：

| 字符串 | 最长相等的真前缀和后缀 | LPS 值 |
| --- | --- | --- |
| `a` | 无 | `0` |
| `ab` | 无 | `0` |
| `aba` | `a` | `1` |
| `abab` | `ab` | `2` |
| `ababa` | `aba` | `3` |

所以对 `pattern = "ababaca"`：

| i | pattern[0:i+1] | lps[i] |
| --- | --- | --- |
| 0 | `a` | 0 |
| 1 | `ab` | 0 |
| 2 | `aba` | 1 |
| 3 | `abab` | 2 |
| 4 | `ababa` | 3 |
| 5 | `ababac` | 0 |
| 6 | `ababaca` | 1 |

因此：

```python
build_lps("ababaca") == [0, 0, 1, 2, 3, 0, 1]
```

假设匹配过程中已经有一段成功匹配：

```text
text:    ... a b a b a x ...
pattern:     a b a b a c
             0 1 2 3 4 5
```

前 5 个字符 `ababa` 匹配成功，但在下一个字符处失配：`x != c`。

这时不需要把模式串完全移到下一位重新比较，因为已匹配片段 `ababa` 内部有结构：

```text
ababa
前缀: aba
后缀: aba
```

也就是说，已经匹配上的后缀 `aba` 可以继续作为下一轮匹配的前缀。模式串指针 `j` 可以从 `5` 回退到 `3`，不需要让主串指针 `i` 回退。

因此，`lps[j - 1]` 的含义正好是：

> 当前已经匹配 `j` 个字符，下一位失配时，模式串指针 `j` 应该回退到哪里。

这就是 KMP 不重复比较主串字符的根本原因。

接下来我们考虑 lps 数组的构造。

现在要对模式串本身做一次“自匹配”。

维护两个变量：

- `i`：当前正在计算 `lps[i]`。
- `length`：当前候选的最长相等前后缀长度，也可以理解为“前缀指针”。

在计算 `lps[i]` 时：

1. 如果 `pattern[i] == pattern[length]`，说明当前相等前后缀可以延长一位：

   ```python
   length += 1
   lps[i] = length
   i += 1
   ```

2. 如果 `pattern[i] != pattern[length]`，但 `length > 0`，说明当前候选长度太长，需要缩短候选前缀。新的候选长度不是简单地 `length - 1`，而是：

   ```python
   length = lps[length - 1]
   ```

   原因是：如果长度为 `length` 的前缀无法继续匹配，那么下一个可能的候选，应该是这个前缀自身的最长相等真前后缀。

3. 如果 `pattern[i] != pattern[length]` 且 `length == 0`，说明没有更短的候选了：

   ```python
   lps[i] = 0
   i += 1
   ```

注意第 2 种情况中，`i` 不移动。因为只是更换候选前缀，当前字符 `pattern[i]` 还没有被处理完。

构造过程的时间复杂度是 `O(m)`。虽然 `length` 有时会回退，但每次回退都对应之前已经增长过的长度，总次数受 `m` 限制。

### LPS 构造实现

In [ ]:
def build_lps(pattern: str) -> list[int]:
    """构造 pattern 的 LPS 数组。"""
    m = len(pattern)
    lps = [0] * m

    length = 0  # 当前候选的最长相等前后缀长度
    i = 1       # lps[0] 一定是 0，因此从 1 开始

    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length > 0:
            length = lps[length - 1]
        else:
            lps[i] = 0
            i += 1

    return lps

In [ ]:
build_lps("ababaca")

## 3. 最终 KMP 算法

KMP 搜索阶段维护两个指针：

- `i`：主串 `text` 的当前位置。
- `j`：模式串 `pattern` 的当前位置，也表示当前已经匹配的模式串长度。

匹配规则：

1. 如果 `text[i] == pattern[j]`，两个指针都向右移动。
2. 如果 `j == len(pattern)`，说明找到一次完整匹配，起点是 `i - j`。为了继续寻找重叠匹配，把 `j` 回退到 `lps[j - 1]`。
3. 如果 `text[i] != pattern[j]` 且 `j > 0`，主串指针 `i` 不动，模式串指针回退到 `lps[j - 1]`。
4. 如果 `text[i] != pattern[j]` 且 `j == 0`，没有可复用的前缀，主串指针 `i` 右移一位。

整个过程中，`i` 永远不会左移。KMP 的复杂度是：

- 预处理 LPS：`O(m)`。
- 搜索：`O(n)`。
- 总时间复杂度：`O(n + m)`。
- 额外空间复杂度：`O(m)`。

### KMP 搜索实现

In [ ]:
def kmp_search(text: str, pattern: str) -> list[int]:
    """返回 pattern 在 text 中出现的所有起始下标，支持重叠匹配。"""
    n, m = len(text), len(pattern)

    if m == 0:
        return list(range(n + 1))

    lps = build_lps(pattern)
    matches = []

    i = 0  # text 指针
    j = 0  # pattern 指针

    while i < n:
        if text[i] == pattern[j]:
            i += 1
            j += 1

            if j == m:
                matches.append(i - j)
                j = lps[j - 1]
        elif j > 0:
            j = lps[j - 1]
        else:
            i += 1

    return matches

In [ ]:
kmp_search("ABABDABACDABABCABAB", "ABABCABAB")

In [ ]:
# 重叠匹配示例
kmp_search("AAAAA", "AAA")